# Product Evidence Platform — Mandatory Product URL, Adaptive Search and RCA

Every successful or review-required run must return a real direct product URL. Strict verification and URL delivery are reported separately: a `REVIEW_REQUIRED` result still contains the strongest real product page, while a run with no direct candidate fails with `MANDATORY_PRODUCT_URL_NOT_FOUND`.

The three SerpAPI credits remain adaptive and follow the internal source hierarchy: **requested retailer → local/regional manufacturer → global manufacturer → major country retailer → other local website → other global website → Amazon/eBay last resort**.


In [ ]:
from __future__ import annotations
import importlib, json, subprocess, sys
from pathlib import Path
from pprint import pprint

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'docker-compose.yml').is_file() and (candidate / 'src' / 'product_evidence_harness').is_dir():
            return candidate
    raise RuntimeError('Could not locate the web_search_tool repository root')

PROJECT_ROOT = find_project_root()
SRC_PATH = (PROJECT_ROOT / 'src').resolve()
LOCAL_PACKAGE = (SRC_PATH / 'product_evidence_harness').resolve()
REQUIRED_MODULE = LOCAL_PACKAGE / 'notebook_runtime.py'
if not REQUIRED_MODULE.is_file():
    raise FileNotFoundError(f'Missing {REQUIRED_MODULE}. Run git checkout master && git pull origin master.')

for required_path in (str(PROJECT_ROOT), str(SRC_PATH)):
    sys.path[:] = [item for item in sys.path if str(Path(item or '.').resolve()) != required_path]
    sys.path.insert(0, required_path)

for module_name in tuple(sys.modules):
    if module_name == 'product_evidence_harness' or module_name.startswith('product_evidence_harness.') or module_name == 'src.product_evidence_harness' or module_name.startswith('src.product_evidence_harness.'):
        del sys.modules[module_name]
importlib.invalidate_caches()

PACKAGE_IMPORTS = {
    'pandas': 'pandas>=2.2,<3',
    'matplotlib': 'matplotlib>=3.8,<4',
    'seaborn': 'seaborn>=0.13.2,<1',
    'rich': 'rich>=13.7,<15',
    'openpyxl': 'openpyxl>=3.1,<4',
}
missing_specs = []
for module_name, package_spec in PACKAGE_IMPORTS.items():
    try:
        importlib.import_module(module_name)
    except ImportError:
        missing_specs.append(package_spec)
if missing_specs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing_specs])

import product_evidence_harness
loaded_package = Path(product_evidence_harness.__file__).resolve()
if LOCAL_PACKAGE not in loaded_package.parents:
    raise RuntimeError(f'Wrong package loaded. Expected {LOCAL_PACKAGE}; loaded {loaded_package}')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from rich.console import Console
from rich.panel import Panel

from product_evidence_harness.notebook_runtime import AGENT_URL, HEARTBEAT_SECONDS, DEFAULT_FEATURE_SET, check_health, run_product, host_artifact_dir
from product_evidence_harness.notebook_diagnostics import build_single_product_diagnostics, display_compact, display_rich_summary, plot_all_diagnostics, plot_candidate_outcomes, plot_confidence_distribution, plot_confidence_vs_coverage, plot_domain_quality, plot_feature_heatmap, plot_funnel, plot_rejection_reasons, plot_stage_yield
from product_evidence_harness.source_authority_notebook import apply_source_authority_notebook_patch
apply_source_authority_notebook_patch()
from product_evidence_harness.adaptive_notebook_diagnostics import build_adaptive_search_diagnostics, display_adaptive_search_summary, export_adaptive_search_tables, plot_credit_progression, plot_engine_candidate_yield, plot_engine_credit_allocation

sns.set_theme(context='notebook', style='whitegrid')
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 140)
pd.set_option('display.width', 240)

health = check_health()
feature_sets = sorted(path.stem for path in (PROJECT_ROOT / 'inputs' / 'private').glob('*.json'))
if DEFAULT_FEATURE_SET not in feature_sets:
    raise RuntimeError('The committed inputs/private/toy_features.json is missing')
Console().print(Panel(
    f'Repository: {PROJECT_ROOT}\nPackage: {loaded_package}\nAgent: {AGENT_URL}\nDefault feature set: {DEFAULT_FEATURE_SET}\nAvailable feature sets: {", ".join(feature_sets)}',
    title='Notebook readiness',
))
pprint(health)


## 1. Run one product

`main_text` and `country_code` are mandatory. `retailer_name`, `ean`, and `language_code` are optional. Keep EAN/GTIN as text.


In [ ]:
FEATURE_SET = 'toy_features'
RUN_SINGLE_PRODUCT = False
product = {
    'row_id': 'ROW-001',
    'main_text': 'Replace with the exact product identity text',
    'country_code': 'CH',
    'retailer_name': None,
    'ean': None,
    'language_code': None,
}
if RUN_SINGLE_PRODUCT:
    if product['main_text'].startswith('Replace with'):
        raise ValueError("Replace product['main_text'] before running")
    result = run_product(product, FEATURE_SET)
    search = result.get('search') or {}
    acceptance = result.get('primary_url_acceptance') or {}
    delivery = result.get('url_delivery') or {}
    if not result.get('primary_url') or not delivery.get('delivered'):
        raise RuntimeError(f'Mandatory URL contract violated: {json.dumps(result, indent=2, default=str)[:4000]}')
    pprint({
        'row_id': (result.get('product') or {}).get('row_id'),
        'job_status': result.get('job_status'),
        'primary_url': result.get('primary_url'),
        'url_delivery_status': delivery.get('status'),
        'strictly_verified': delivery.get('strictly_verified'),
        'strict_primary_accepted': acceptance.get('accepted'),
        'engine_sequence': search.get('engine_sequence'),
        'target_source_tiers': search.get('target_source_tiers'),
        'serpapi_requests_used': search.get('serpapi_requests_used'),
        'search_stop_reason': search.get('stop_reason'),
    })
else:
    print('Ready. Replace the product input and set RUN_SINGLE_PRODUCT = True.')


## 2. Build the complete diagnostic model

`serp_results_df` preserves raw result occurrences. `results_df` contains one canonical URL per row. `url_delivery_df` records the mandatory URL outcome independently from strict acceptance.


In [ ]:
if 'result' not in globals():
    raise RuntimeError('Run the single-product cell first')
artifact_path = host_artifact_dir(PROJECT_ROOT, result)
if artifact_path is None or not artifact_path.is_dir():
    raise RuntimeError('Repository-local artifact directory was not found')

diagnostics = build_single_product_diagnostics(result, artifact_dir=artifact_path)
adaptive_diagnostics = build_adaptive_search_diagnostics(result)
overview_df = diagnostics.overview_df
search_stages_df = diagnostics.search_stages_df
serp_results_df = diagnostics.serp_results_df
results_df = diagnostics.results_df
agentic_df = diagnostics.agentic_df
feature_evidence_df = diagnostics.feature_evidence_df
feature_matrix_df = diagnostics.feature_matrix_df
funnel_df = diagnostics.funnel_df
domain_summary_df = diagnostics.domain_summary_df
stage_quality_df = diagnostics.stage_quality_df
rejection_reasons_df = diagnostics.rejection_reasons_df
selection_rca_df = diagnostics.selection_rca_df
search_actions_df = adaptive_diagnostics.search_actions_df
search_engine_summary_df = adaptive_diagnostics.search_engine_summary_df
search_handles_df = adaptive_diagnostics.search_handles_df
search_decision_rca_df = adaptive_diagnostics.search_decision_rca_df
source_hierarchy_df = search_actions_df[[column for column in ('serp_credit', 'target_source_tier', 'engine', 'purpose', 'planner_source', 'results_returned', 'new_candidate_urls', 'candidates_qualified', 'candidates_scraped', 'working_url_found', 'reason', 'query') if column in search_actions_df]].copy()
source_tier_summary_df = (
    results_df.groupby(['source_tier', 'source_tier_name', 'source_role'], dropna=False)
    .agg(candidate_urls=('url', 'count'), scrape_attempted=('scrape_attempted', 'sum'), scrape_accepted=('scrape_success', 'sum'), identity_accepted=('identity_accepted', 'sum'), selected_urls=('strict_selected', 'sum'), mean_confidence=('confidence', 'mean'))
    .reset_index().sort_values(['source_tier', 'mean_confidence'], ascending=[True, False])
    if not results_df.empty and 'source_tier' in results_df else pd.DataFrame()
)
url_delivery_df = pd.DataFrame([{
    **(result.get('url_delivery') or {}),
    'job_status': result.get('job_status'),
    'strict_acceptance': (result.get('primary_url_acceptance') or {}).get('accepted'),
    'primary_url': result.get('primary_url'),
}])
display_rich_summary(diagnostics)
display_adaptive_search_summary(adaptive_diagnostics)


## 3. Mandatory URL, search route and candidate RCA


In [ ]:
display_compact(url_delivery_df, title='Mandatory product URL delivery', max_rows=5)
display_compact(source_hierarchy_df, title='Source hierarchy by SerpAPI credit', max_rows=10)
display_compact(search_actions_df, title='Adaptive SerpAPI decisions', max_rows=10)
display_compact(search_engine_summary_df, title='Search-engine yield and conversion', max_rows=20)
display_compact(search_decision_rca_df, title='Search decision RCA', max_rows=40)
display_compact(search_handles_df, title='Product tokens, identifiers and image handles', max_rows=30)
display_compact(source_tier_summary_df, title='Candidate conversion by source tier', max_rows=20)
display_compact(results_df, title='One canonical product URL candidate per row', max_rows=100)
assert results_df['url'].is_unique
display_compact(selection_rca_df, title='Final URL selection RCA', max_rows=30)
display_compact(funnel_df, title='Candidate conversion funnel', max_rows=20)
display_compact(rejection_reasons_df, title='Rejection and review reasons', max_rows=40)
display_compact(agentic_df, title='Agentic-browser investigations', max_rows=30)
display_compact(feature_evidence_df, title='URL-feature evidence', max_rows=100)


## 4. Graphical diagnostics


In [ ]:
plot_engine_credit_allocation(adaptive_diagnostics)
plot_engine_candidate_yield(adaptive_diagnostics)
plot_credit_progression(adaptive_diagnostics)
plot_funnel(diagnostics)
plot_stage_yield(diagnostics)
plot_candidate_outcomes(diagnostics)
plot_confidence_distribution(diagnostics)
plot_confidence_vs_coverage(diagnostics)
plot_domain_quality(diagnostics)
plot_rejection_reasons(diagnostics)
plot_feature_heatmap(diagnostics)
plot_all_diagnostics(diagnostics)


## 5. Export the diagnostic workbook


In [ ]:
workbook_path = artifact_path / 'single_product_diagnostics.xlsx'
with pd.ExcelWriter(workbook_path, engine='openpyxl', mode='w') as writer:
    for name, frame in diagnostics.tables().items():
        frame.to_excel(writer, sheet_name=name.removesuffix('_df')[:31], index=False)
    source_tier_summary_df.to_excel(writer, sheet_name='source_tier_summary', index=False)
    url_delivery_df.to_excel(writer, sheet_name='url_delivery', index=False)
export_adaptive_search_tables(adaptive_diagnostics, workbook_path)
print(f'Diagnostic workbook: {workbook_path}')
print(f'Mandatory URL artifact: {artifact_path / "mandatory_url_delivery.json"}')
print(f'Adaptive trace: {artifact_path / "adaptive_search_trace.json"}')
